# Vérifications et exemples, avec et sans licence ecoinvent

[English](check_and_examples.ipynb) | **Français**

Ce notebook rassemble les vérifications et les exemples faits sur le dépôt :

- **Partie 1, avec une licence ecoinvent** : importer ecoinvent 3.11 cut-off dans Brightway, recalculer les impacts de fabrication et vérifier qu'ils sont identiques à ceux fournis (calculés avec Activity Browser) dans les 16 catégories EF 3.1 ; puis un exemple de changement d'inventaire.
- **Partie 2, sans licence** : les résultats de référence et trois modifications du modèle.

Les valeurs attendues sont données sous chaque cellule. Lancez les cellules dans l'ordre. Dans les commandes, `!` lance une commande du terminal et `{sys.executable}` désigne le Python du notebook.

## 0. Télécharger le dépôt et installer (une fois)

Sans Git : le dépôt est téléchargé en ZIP dans votre dossier personnel. La deuxième ligne `pip` n'est utile que pour la Partie 1.

In [ ]:
import sys, os, urllib.request, zipfile
os.chdir(os.path.expanduser("~"))
urllib.request.urlretrieve("https://github.com/Ayabh12/h2-ageing-lca/archive/refs/heads/main.zip", "h2-ageing-lca.zip")
zipfile.ZipFile("h2-ageing-lca.zip").extractall(".")
os.chdir("h2-ageing-lca-main")
!{sys.executable} -m pip install -r requirements.txt
!{sys.executable} -m pip install -U -r requirements-brightway.txt

Après l'installation de Brightway, **redémarrez le noyau** (menu *Kernel → Restart*), puis continuez.

# Partie 1 – Vérification du calcul de fabrication avec ecoinvent

Nécessite une licence ecoinvent (la vôtre ou celle de votre établissement, si ses conditions le permettent). L'identifiant n'est écrit dans aucun fichier : ne le tapez pas dans une cellule que vous partagez.

### 1.1 Identifiant ecoinvent

Le mot de passe ne s'affiche pas pendant la saisie.

In [ ]:
import sys, os
import pandas as pd
os.chdir(os.path.expanduser("~/h2-ageing-lca-main"))
import getpass
os.environ["EI_USERNAME"] = input("Identifiant ecoinvent : ")
os.environ["EI_PASSWORD"] = getpass.getpass("Mot de passe ecoinvent : ")

### 1.2 Importer ecoinvent et recalculer les impacts de fabrication

La première fois, ecoinvent 3.11 cut-off est téléchargé et importé dans le projet Brightway `h2ageing` (de 20 minutes à moins d'une heure). Les fois suivantes, le calcul prend environ une minute.

**Attendu** : la liste des 16 méthodes EF 3.1 utilisées, puis le changement climatique de chaque composant : `fuel_cell_stack_128kW` 3698.3, `electrolyser_stack_1p25MW` 39509.4, `tank_type_IV` 2332.0 kg CO2-éq.

In [ ]:
!{sys.executable} scripts/build_fabrication_brightway.py --project h2ageing --import-ecoinvent --out results/fabrication_test.csv

### 1.3 Vérification des 16 catégories EF 3.1

Écart (en %) entre les impacts recalculés avec Brightway et ceux d'Activity Browser fournis dans le dépôt.

**Attendu** : 0 partout (−0 est un arrondi) ; écart inférieur à 0,01 % dans les 16 catégories et pour les trois composants.

In [ ]:
nouveau = pd.read_csv("results/fabrication_test.csv").set_index("category")
reference = pd.read_csv("data/impacts/fabrication_per_unit.csv").set_index("category")
ecart = ((nouveau - reference) / reference * 100).round(2)
print("Écart maximal :", ecart.abs().max().max(), "%")
ecart

### 1.4 Exemple : fabriquer le stack de pile avec l'électricité japonaise

Le stack de la Mirai est fabriqué par Toyota au Japon. On remplace le mix résiduel français par le marché de l'électricité haute tension du Japon (ecoinvent 3.11) pour les 1 096 kWh de fabrication du stack, puis on relance tout le modèle. L'inventaire d'origine est remis en place à la fin de la cellule.

**Attendu** : impact d'un stack de pile d'environ 4 316 kg CO2-éq au lieu de 3 698 ; chaîne (g CO2-éq/km) : sans vieillissement 74,1 → 77,2 ; central 97,2 → 103,3 ; hausse due au vieillissement dans le cas central +31,1 % → +33,8 %.

In [ ]:
p = "data/inventories/fuel_cell_stack_128kW.csv"
origine = open(p, encoding="utf-8", newline="").read()
inv = pd.read_csv(p, dtype=str, keep_default_na=False)
ligne = inv["input"].str.contains("French residual mix")
inv.loc[ligne, "input"] = "Electricity, Japan, high voltage"
inv.loc[ligne, "ecoinvent_activity"] = "market for electricity, high voltage"
inv.loc[ligne, "location"] = "JP"
inv.to_csv(p, index=False)
try:
    !{sys.executable} scripts/build_fabrication_brightway.py --project h2ageing --out results/fabrication_japon.csv
finally:
    open(p, "w", encoding="utf-8", newline="").write(origine)   # inventaire d'origine remis en place

!{sys.executable} scripts/run.py --out results/reference > {os.devnull}
!{sys.executable} scripts/run.py --set fabrication.file=results/fabrication_japon.csv --out results/japon > {os.devnull}
ref = pd.read_csv("results/reference/chain.csv").set_index("case")
jp = pd.read_csv("results/japon/chain.csv").set_index("case")
pd.DataFrame({"France : total (g/km)": ref["total_g_per_km"], "Japon : total (g/km)": jp["total_g_per_km"],
              "France : hausse (%)": ref["rise_pct"], "Japon : hausse (%)": jp["rise_pct"]}).round(1)

# Partie 2 – Utiliser et modifier le modèle sans ecoinvent

Le modèle utilise les impacts de fabrication fournis dans le dépôt (`data/impacts`). Aucune licence n'est nécessaire. Ces résultats sont aussi vérifiés automatiquement par `tests/test_reference_results.py`.

In [ ]:
import sys, os
import pandas as pd
os.chdir(os.path.expanduser("~/h2-ageing-lca-main"))

## Exemple 1 – Résultats de référence

**Attendu** (g CO2-éq/km) : sans vieillissement 74,1 ; optimiste 77,8 (+5,0 %) ; central 97,2 (+31,1 %) ; pessimiste 100,1 (+35,0 %).

In [ ]:
!{sys.executable} scripts/run.py --out results/reference > {os.devnull}
ref = pd.read_csv("results/reference/chain.csv").set_index("case")
ref[["total_g_per_km", "rise_pct"]].round(1)

## Exemple 2 – Modifier la durée de vie de la pile : 6 000 h au lieu de 4 100 h

**Attendu**, cas central : 1 stack de pile au lieu de 2 ; chaîne 97,2 → 78,9 g/km ; hausse +31,1 % → +6,4 %. Au-delà de 5 554 h, un seul stack suffit sur toute la vie du véhicule.

In [ ]:
!{sys.executable} scripts/run.py --set cases.central.fuel_cell.hours_to_end_of_life=6000 --out results/pile_6000h > {os.devnull}
vie = pd.read_csv("results/pile_6000h/life.csv").set_index("case")
p6 = pd.read_csv("results/pile_6000h/chain.csv").set_index("case")
display(vie[["fuel_cell_end_of_life_h", "fuel_cell_stacks"]].round(0))
pd.DataFrame({"Référence (g/km)": ref["total_g_per_km"], "Pile 6 000 h (g/km)": p6["total_g_per_km"],
              "Hausse référence (%)": ref["rise_pct"], "Hausse 6 000 h (%)": p6["rise_pct"]}).round(1)

## Exemple 3 – Électricité éolienne pour l'électrolyseur

**Attendu** : sans vieillissement 74,1 → 41,2 g/km ; central 97,2 → 61,3 g/km ; hausse due au vieillissement +31,1 % → +48,6 %. Avec une électricité propre, les stacks de remplacement pèsent plus en proportion. Autres choix : `average_mix_FR`, `nuclear_FR`, `photovoltaic`.

In [ ]:
!{sys.executable} scripts/run.py --set usage.electricity_electrolyser=wind_FR --out results/eolien > {os.devnull}
eo = pd.read_csv("results/eolien/chain.csv").set_index("case")
pd.DataFrame({"Mix résiduel (g/km)": ref["total_g_per_km"], "Éolien (g/km)": eo["total_g_per_km"],
              "Hausse mix résiduel (%)": ref["rise_pct"], "Hausse éolien (%)": eo["rise_pct"]}).round(1)

## Exemple 4 – Utiliser sa propre valeur d'impact de fabrication

Valeur de test : stack de pile 30 % moins émetteur. Avec une valeur tirée d'une autre étude, remplacez `*= 0.7` par `= <valeur>` (kg CO2-éq par stack).

**Attendu** : sans vieillissement 74,1 → 68,6 g/km ; central 97,2 → 86,1 g/km ; hausse +31,1 % → +25,5 %.

In [ ]:
fab = pd.read_csv("data/impacts/fabrication_per_unit.csv").set_index("category")
fab.loc["climate_change", "fuel_cell_stack_128kW"] *= 0.7
os.makedirs("results", exist_ok=True)
fab.to_csv("results/ma_fabrication.csv")
!{sys.executable} scripts/run.py --set fabrication.file=results/ma_fabrication.csv --out results/ma_fabrication > {os.devnull}
mf = pd.read_csv("results/ma_fabrication/chain.csv").set_index("case")
pd.DataFrame({"Référence (g/km)": ref["total_g_per_km"], "Stack −30 % (g/km)": mf["total_g_per_km"],
              "Hausse référence (%)": ref["rise_pct"], "Hausse stack −30 % (%)": mf["rise_pct"]}).round(1)

## Autres exemples à tester

Les exemples suivants changent les modèles eux-mêmes : une constante, une loi, le modèle d'un cas, ou un cas entier. La fonction `comparer` met côte à côte la référence et la variante (chaîne complète, g CO2-éq/km, et hausse due au vieillissement). Lancez d'abord l'exemple 1, qui calcule la référence `ref`.

In [ ]:
import sys
sys.path.insert(0, "src")
from h2ageing import load_config, run

def comparer(resultats, nom):
    ch = resultats["chain"].set_index("case")
    vie = resultats["life"].set_index("case")
    return pd.DataFrame({
        "Référence (g/km)": ref["total_g_per_km"], f"{nom} (g/km)": ch["total_g_per_km"],
        "Hausse référence (%)": ref["rise_pct"], f"Hausse {nom} (%)": ch["rise_pct"],
        "Fin de vie pile (h)": vie["fuel_cell_end_of_life_h"], "Stacks de pile": vie["fuel_cell_stacks"],
    }).round(1)

## Exemple 5 – Changer une constante du modèle semi-empirique

Dans le cas pessimiste, la perte de tension de la pile à fort courant (1,8 A/cm², après 1 000 h) passe de 45 mV à 27 mV, la valeur publiée pour une autre cellule (voir `scripts/semi_empirical_constants.py`).

**Attendu**, cas pessimiste : fin de vie de la pile 2 895 → 5 652 h, 1 stack au lieu de 2 ; chaîne 100,1 → 83,6 g/km ; hausse +35,0 % → +12,8 %. C'est la constante du modèle semi-empirique qui pèse le plus.

In [ ]:
cfg = load_config()
cfg["cases"]["pessimistic"]["fuel_cell"]["voltage_loss_at_1p8_A_cm2_1000h_V"] = 0.027
comparer(run(cfg), "27 mV")

## Exemple 6 – Changer une loi : crossover exponentiel au lieu de linéaire

Dans le modèle semi-empirique de la pile, le crossover d'hydrogène à travers la membrane augmente par défaut de façon linéaire. On le fait croître de façon exponentielle (autre choix : `"constant"`).

**Attendu**, cas pessimiste : la fin de vie ne change pas (2 895 h, le critère porte sur la tension) ; la surconsommation augmente, et la hausse passe de +35,0 % à +36,9 % (+32,6 % avec un crossover constant). La loi de crossover compte peu devant la durée de vie.

In [ ]:
cfg = load_config()
cfg["fuel_cell"]["crossover_law"] = "exponential"
comparer(run(cfg), "exponentiel")

## Exemple 7 – Changer de modèle : linéaire au lieu de semi-empirique, à fin de vie égale

Dans le cas pessimiste, on remplace le modèle semi-empirique de la pile par le modèle linéaire, réglé sur la même fin de vie (2 895 h). Seule la forme de la dégradation dans le temps change.

**Attendu**, cas pessimiste : 2 stacks dans les deux cas ; chaîne 100,1 → 99,1 g/km ; hausse +35,0 % → +33,7 %. À fin de vie égale, le choix du modèle change peu le résultat : c'est la fin de vie prédite qui compte.

In [ ]:
cfg = load_config()
cfg["cases"]["pessimistic"]["fuel_cell"] = {"model": "linear", "hours_to_end_of_life": 2895}
comparer(run(cfg), "linéaire 2 895 h")

## Exemple 8 – Créer son propre cas de vieillissement

On ajoute un quatrième cas, avec ses propres valeurs : pile de 6 000 h, électrolyseur à 3,0 µV/h par cellule, perméation du réservoir de 6 à 10 mL/h/L. Ce sont des valeurs de test : remplacez-les par les vôtres.

**Attendu**, cas `mon_cas` : 1 stack de pile, 2 stacks d'électrolyseur ; chaîne 78,3 g/km ; hausse +5,6 %.

In [ ]:
cfg = load_config()
cfg["cases"]["mon_cas"] = {
    "fuel_cell": {"model": "linear", "hours_to_end_of_life": 6000},
    "electrolyser": {"model": "linear", "rate_uV_per_h_per_cell": 3.0},
    "tank_permeation_mL_h_L": [6.0, 10.0],
}
res = run(cfg)
pd.concat([res["chain"].set_index("case")[["total_g_per_km", "rise_pct"]],
           res["life"].set_index("case")[["fuel_cell_stacks", "electrolyser_stacks"]]], axis=1).round(1)

## Exemple 9 – Changer le kilométrage : 150 000 km au lieu de 200 000 km

**Attendu** : la fabrication est répartie sur moins de kilomètres, donc l'impact par km augmente (sans vieillissement 74,1 → 84,2 g/km ; central 97,2 → 113,9 g/km) ; la hausse centrale passe de +31,1 % à +35,2 %, car il faut toujours 2 stacks de pile.

In [ ]:
cfg = load_config()
cfg["horizon"]["vehicle_km_total"] = 150000
comparer(run(cfg), "150 000 km")